# Week 5 lab — Introduction to Machine Learning and Logistic Regression

This notebook accompanies **Lecture 5** of *Machine Learning in Finance*.

It first establishes the distinction between a theory-led econometric question and a prediction-led machine-learning question. It then builds a transparent probability-of-default model using logistic regression. The sample is synthetic/course-sized and is for learning the workflow, not for making lending decisions.


## Learning objectives

By the end of the lab you should be able to:

- define one row, the target, horizon, and information cutoff;
- calculate a logistic probability from a linear score;
- explain likelihood, log-loss, and regularisation;
- compare Ridge, LASSO, and Elastic Net with cross-validation;
- separate ranking quality, probability quality, and decision quality;
- separate probability estimation, model evaluation, and threshold policy.


## Machine learning and econometrics

Econometrics often begins with an economic mechanism, a functional form, and assumptions that support interpretation or inference. Machine learning begins with a prediction task and uses a validation design to estimate how well a fitted rule generalises to new observations. The same regression mathematics can serve either purpose. The research question, loss function, and evidence used for model choice determine the role.

## Learning language for this notebook

A **feature** is an input variable, **target** is the outcome to predict, **label** is the observed target value, **parameter** is fitted from data, and **hyperparameter** is selected by validation. In the binary credit example, `default_next_month = 1` is the positive label. The information cutoff must precede the outcome window.

## Week 5 route

The notebook follows one route: define the financial question, protect the data boundary, compute the score and sigmoid, derive likelihood and log-loss, compare regularisation choices, interpret coefficients, and separate probability estimation from threshold policy. Trees, boosting, clustering, reinforcement learning, explainability, and neural networks belong to later weeks.

In [1]:
from pathlib import Path
import os
import warnings
os.environ.setdefault("LOKY_MAX_CPU_COUNT", "1")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.calibration import calibration_curve
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, average_precision_score, brier_score_loss,
    confusion_matrix, log_loss, precision_score, recall_score,
    roc_auc_score, RocCurveDisplay, PrecisionRecallDisplay,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
warnings.filterwarnings("ignore")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({"figure.figsize": (8, 4.5), "axes.titlesize": 13})
print("Environment ready. Seed:", RANDOM_STATE)


Environment ready. Seed: 42


## 1. Define the financial question

One observation is a borrower snapshot at a decision time t. The target is a
future event, not a current feature:

> At snapshot time t, estimate the probability that the borrower defaults in the
> next outcome window using only information available by t.

For this lab, default_next_month = 1 is the positive class and ID is an identifier.


In [2]:
DATA_CANDIDATES = [
    Path("data/credit_ml_sample.csv"),
    Path("data/credit_ml_sample.csv"),
    Path("credit_feature_engineering_sample.csv"),
]
data_path = next((p for p in DATA_CANDIDATES if p.exists()), None)
if data_path is not None:
    df = pd.read_csv(data_path)
    source_note = str(data_path)
else:
    rng = np.random.default_rng(RANDOM_STATE)
    n = 1000
    utilization = rng.beta(2.0, 5.0, size=n)
    overdue = rng.poisson(0.8, size=n)
    income = rng.lognormal(10.3, 0.65, size=n)
    score = -2.3 + 2.2 * utilization + 0.35 * overdue - 0.000002 * income
    probability = 1 / (1 + np.exp(-score))
    df = pd.DataFrame({
        "ID": np.arange(1, n + 1),
        "utilization": utilization,
        "overdue_count_6m": overdue,
        "annual_income": income,
        "default_next_month": rng.binomial(1, probability),
    })
    source_note = "deterministic fallback generated inside the notebook"
TARGET = "default_next_month"
print("Data source:", source_note)
print("Shape:", df.shape)
print("Target rate:", round(df[TARGET].mean(), 4))
display(df.head())


Data source: data/credit_ml_sample.csv
Shape: (1000, 20)
Target rate: 0.214


In [3]:
audit = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing_n": df.isna().sum(),
    "missing_pct": (100 * df.isna().mean()).round(2),
    "n_unique": df.nunique(),
})
print("Target counts:")
print(df[TARGET].value_counts().sort_index().to_string())
print("\nAudit:")
print(audit.to_string())
fig, ax = plt.subplots()
df[TARGET].value_counts().sort_index().plot(kind="bar", ax=ax, color=["#0F8795", "#D26455"])
ax.set_title("Course sample: target balance")
ax.set_xlabel("Default next month (0 = no, 1 = yes)")
ax.set_ylabel("Number of observations")
plt.tight_layout()
plt.show()


Target counts:
default_next_month
0    786
1    214

Audit:
                      dtype  missing_n  missing_pct  n_unique
ID                    int64          0          0.0      1000
LIMIT_BAL             int64          0          0.0        56
PAY_0                 int64          0          0.0         8
BILL_AMT1             int64          0          0.0       904
BILL_AMT2             int64          0          0.0       875
PAY_AMT1              int64          0          0.0       521
PAY_AMT2              int64          0          0.0       520
default_next_month    int64          0          0.0         2
utilization         float64          0          0.0       923
total_bill_6m         int64          0          0.0       961
total_paid_6m         int64          0          0.0       886
payment_to_bill_6m  float64         35          3.5       912
overdue_count_6m      int64          0          0.0         7
max_delay_6m          int64          0          0.0        11
recent_ove

## 2. Split the data and seal the test sample

Training rows fit transformations and coefficients. Validation rows select the
penalty, model, and threshold. The final test sample is used once, after choices
are frozen. For a genuinely time-indexed portfolio, use an out-of-time split.


In [4]:
feature_cols = [c for c in df.columns if c not in {TARGET, "ID"}]
X = df[feature_cols].select_dtypes(include=np.number).copy()
y = df[TARGET].astype(int).copy()
numeric_cols = X.columns.tolist()
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.40, stratify=y, random_state=RANDOM_STATE
)
X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=RANDOM_STATE
)
split_summary = pd.DataFrame({
    "rows": [len(X_train), len(X_valid), len(X_test)],
    "default_rate": [y_train.mean(), y_valid.mean(), y_test.mean()],
}, index=["train", "validation", "test"])
print(split_summary.round(4).to_string())


            rows  default_rate
train        600        0.2133
validation   200        0.2150
test         200        0.2150


## 3. Logistic probability: score, sigmoid, odds

For observation i, the linear score and sigmoid are

$$
\eta_i = \beta_0 + x_i^{\mathrm T} \beta,
$$

$$
p_i = P(Y_i = 1 \mid x_i) = \frac{1}{1 + e^{-\eta_i}}.
$$

The odds are p_i divided by 1 minus p_i, and the log-odds equal the score.
The coefficient sign describes the direction of the feature on the log-odds,
holding other features fixed.


In [5]:
beta_0 = -1.20
beta_utilization = 1.80
beta_overdue = 0.35
beta_payment_ratio = -0.60
borrower = {"utilization": 0.70, "overdue_count": 2, "payment_ratio": 0.40}
eta = (beta_0 + beta_utilization * borrower["utilization"]
       + beta_overdue * borrower["overdue_count"]
       + beta_payment_ratio * borrower["payment_ratio"])
p_default = 1 / (1 + np.exp(-eta))
odds = p_default / (1 - p_default)
print(f"eta = {eta:.3f}")
print(f"p(default) = {p_default:.3f}")
print(f"odds(default) = {odds:.3f}")


eta = 0.520
p(default) = 0.627
odds(default) = 1.682


## 4. Likelihood, log-loss, and gradient

For a binary observation, the joint likelihood and log-likelihood are

$$
L(\beta) = \prod_{i=1}^{n} p_i^{y_i}(1-p_i)^{1-y_i},
$$

$$
\ell(\beta) = \sum_{i=1}^{n} [y_i\log(p_i) + (1-y_i)\log(1-p_i)].
$$

Training minimises negative average log-likelihood:

$$
J(\beta) = -\frac{1}{n} \sum_{i=1}^{n} [y_i\log(p_i) + (1-y_i)\log(1-p_i)].
$$

For feature j, the unregularised gradient is

$$
\frac{\partial J}{\partial \beta_j} = \frac{1}{n} \sum_{i=1}^{n} (p_i-y_i)x_{ij}.
$$

In a notebook, the symbols should be followed by a numerical check, not left as
unexplained decoration.


In [6]:
def evaluate_binary(model, X_eval, y_eval, label, threshold=0.50):
    p = model.predict_proba(X_eval)[:, 1]
    pred = (p >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_eval, pred, labels=[0, 1]).ravel()
    return {
        "model": label, "threshold": threshold,
        "accuracy": accuracy_score(y_eval, pred),
        "precision": precision_score(y_eval, pred, zero_division=0),
        "recall": recall_score(y_eval, pred, zero_division=0),
        "roc_auc": roc_auc_score(y_eval, p),
        "pr_auc": average_precision_score(y_eval, p),
        "brier": brier_score_loss(y_eval, p),
        "log_loss": log_loss(y_eval, p, labels=[0, 1]),
        "tn": tn, "fp": fp, "fn": fn, "tp": tp,
    }
baseline = DummyClassifier(strategy="prior")
baseline.fit(X_train, y_train)
print(pd.DataFrame([evaluate_binary(baseline, X_valid, y_valid, "prior baseline")]).round(4).to_string(index=False))


         model  threshold  accuracy  precision  recall  roc_auc  pr_auc  brier  log_loss  tn  fp  fn  tp
prior baseline        0.5     0.785        0.0     0.0      0.5   0.215 0.1688    0.5205 157   0  43   0


## 5. Compare Ridge, LASSO, and Elastic Net

The penalty changes the objective, not the information in the data:

$$
J_{\mathrm{ridge}} = J + \lambda \sum_j \beta_j^2,
$$

$$
J_{\mathrm{lasso}} = J + \lambda \sum_j |\beta_j|,
$$

$$
J_{\mathrm{EN}} = J + \lambda [\alpha \sum_j |\beta_j| +
(1-\alpha) \frac{1}{2} \sum_j \beta_j^2].
$$

C in scikit-learn is inverse penalty strength: smaller C means stronger
regularisation. l1_ratio is alpha in Elastic Net.


In [7]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
candidate_specs = {
    "Ridge": (
        Pipeline([("impute", SimpleImputer(strategy="median")),
                  ("scale", StandardScaler()),
                  ("model", LogisticRegression(penalty="l2", solver="lbfgs", max_iter=2000, random_state=RANDOM_STATE))]),
        {"model__C": [0.03, 0.10, 0.30, 1.00, 3.00, 10.00]},
    ),
    "LASSO": (
        Pipeline([("impute", SimpleImputer(strategy="median")),
                  ("scale", StandardScaler()),
                  ("model", LogisticRegression(penalty="l1", solver="liblinear", max_iter=2000, random_state=RANDOM_STATE))]),
        {"model__C": [0.03, 0.10, 0.30, 1.00, 3.00, 10.00]},
    ),
    "Elastic Net": (
        Pipeline([("impute", SimpleImputer(strategy="median")),
                  ("scale", StandardScaler()),
                  ("model", LogisticRegression(penalty="elasticnet", solver="saga", max_iter=3000, random_state=RANDOM_STATE))]),
        {"model__C": [0.03, 0.10, 0.30, 1.00, 3.00], "model__l1_ratio": [0.20, 0.50, 0.80]},
    ),
}
searches = {}
for name, (pipeline, grid) in candidate_specs.items():
    search = GridSearchCV(pipeline, grid, scoring="roc_auc", cv=cv, n_jobs=1, refit=True, return_train_score=True)
    search.fit(X_train, y_train)
    searches[name] = search
selection_table = pd.DataFrame([
    {"model": name, "best_cv_auc": search.best_score_, "best_params": str(search.best_params_)}
    for name, search in searches.items()
]).sort_values("best_cv_auc", ascending=False)
print(selection_table.round(4).to_string(index=False))


      model  best_cv_auc                               best_params
      Ridge       0.7325                        {'model__C': 0.03}
Elastic Net       0.7306 {'model__C': 0.1, 'model__l1_ratio': 0.2}
      LASSO       0.7293                         {'model__C': 0.3}


In [8]:
evaluation_rows = []
probabilities = {}
for name, search in searches.items():
    model = search.best_estimator_
    probabilities[name] = {
        "train": model.predict_proba(X_train)[:, 1],
        "valid": model.predict_proba(X_valid)[:, 1],
        "test": model.predict_proba(X_test)[:, 1],
    }
    evaluation_rows += [
        evaluate_binary(model, X_train, y_train, f"{name} / train"),
        evaluate_binary(model, X_valid, y_valid, f"{name} / validation"),
        evaluate_binary(model, X_test, y_test, f"{name} / test"),
    ]
evaluation_table = pd.DataFrame(evaluation_rows)
print(evaluation_table[["model", "roc_auc", "pr_auc", "brier", "log_loss", "precision", "recall"]].round(4).to_string(index=False))


                   model  roc_auc  pr_auc  brier  log_loss  precision  recall
           Ridge / train   0.7571  0.5098 0.1390    0.4415     0.7045  0.2422
      Ridge / validation   0.7252  0.4742 0.1441    0.4868     0.5806  0.4186
            Ridge / test   0.6806  0.3726 0.1589    0.4918     0.4000  0.1395
           LASSO / train   0.7604  0.5183 0.1383    0.4398     0.6667  0.2969
      LASSO / validation   0.7234  0.4550 0.1461    0.4914     0.5882  0.4651
            LASSO / test   0.6752  0.3663 0.1608    0.4965     0.3684  0.1628
     Elastic Net / train   0.7593  0.5164 0.1382    0.4395     0.7037  0.2969
Elastic Net / validation   0.7230  0.4650 0.1450    0.4921     0.5758  0.4419
      Elastic Net / test   0.6807  0.3713 0.1601    0.4951     0.3684  0.1628


In [9]:
coefficient_rows = []
for name, search in searches.items():
    fitted = search.best_estimator_.named_steps["model"]
    for feature, coefficient in zip(numeric_cols, fitted.coef_[0]):
        coefficient_rows.append({
            "model": name, "feature": feature, "coefficient": coefficient,
            "odds_ratio_for_1_sd": np.exp(coefficient),
            "non_zero": abs(coefficient) > 1e-8,
        })
coefficient_table = pd.DataFrame(coefficient_rows)
display(coefficient_table.reindex(coefficient_table["coefficient"].abs().sort_values(ascending=False).index).head(18).round(3))


## 6. Coefficient paths and sparsity

Ridge usually keeps correlated features together and shrinks coefficients. LASSO
can set coefficients exactly to zero, which makes it useful for sparse selection
but can be unstable with correlated predictors. The plot below is an experiment;
the final C still comes from validation.


In [10]:
C_grid = np.logspace(-2, 1, 10)
path_rows = []
for c_value in C_grid:
    for penalty, solver in [("l1", "liblinear"), ("l2", "lbfgs")]:
        path_model = Pipeline([
            ("impute", SimpleImputer(strategy="median")),
            ("scale", StandardScaler()),
            ("model", LogisticRegression(penalty=penalty, C=c_value, solver=solver, max_iter=2000, random_state=RANDOM_STATE)),
        ])
        path_model.fit(X_train, y_train)
        coefs = path_model.named_steps["model"].coef_[0]
        path_rows.append({"C": c_value, "penalty": penalty,
                          "coefficient_l1_norm": np.abs(coefs).sum(),
                          "non_zero": (np.abs(coefs) > 1e-8).sum()})
path_table = pd.DataFrame(path_rows)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for penalty, color in [("l1", "#D26455"), ("l2", "#0F8795")]:
    part = path_table[path_table["penalty"] == penalty]
    axes[0].plot(part["C"], part["coefficient_l1_norm"], marker="o", label=penalty.upper(), color=color)
    axes[1].plot(part["C"], part["non_zero"], marker="o", label=penalty.upper(), color=color)
axes[0].set_xscale("log"); axes[0].set_title("Coefficient L1 norm")
axes[1].set_xscale("log"); axes[1].set_title("Number of non-zero coefficients")
for ax in axes:
    ax.set_xlabel("C (inverse penalty strength)"); ax.legend()
plt.tight_layout(); plt.show()


## 7. Thresholds are policies

The model estimates a probability. A policy chooses a threshold c:

$$
\widehat{y}_i(c) = \begin{cases} 1 & \text{if } p_i \ge c \\ 0 & \text{if } p_i < c. \end{cases}
$$

Changing c changes false positives, false negatives, approval volume, and review
volume without refitting the coefficients. The correct threshold depends on loss,
capacity, regulation, and customer treatment.


In [11]:
best_name = selection_table.iloc[0]["model"]
p_valid = probabilities[best_name]["valid"]
threshold_rows = []
for threshold in np.arange(0.10, 0.91, 0.10):
    pred = (p_valid >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_valid, pred, labels=[0, 1]).ravel()
    threshold_rows.append({
        "threshold": threshold, "positive_decision_pct": pred.mean(),
        "precision": precision_score(y_valid, pred, zero_division=0),
        "recall": recall_score(y_valid, pred, zero_division=0),
        "false_positives": fp, "false_negatives": fn,
    })
threshold_table = pd.DataFrame(threshold_rows)
print("Selected by CV:", best_name)
print(threshold_table.round(3).to_string(index=False))


Selected by CV: Ridge
 threshold  positive_decision_pct  precision  recall  false_positives  false_negatives
       0.1                  0.895      0.218   0.907              140                4
       0.2                  0.380      0.395   0.698               46               13
       0.3                  0.265      0.509   0.628               26               16
       0.4                  0.220      0.568   0.581               19               18
       0.5                  0.155      0.581   0.419               13               25
       0.6                  0.050      0.700   0.163                3               36
       0.7                  0.030      0.500   0.070                3               40
       0.8                  0.020      0.500   0.047                2               41
       0.9                  0.010      0.000   0.000                2               43


In [12]:
c_fn, c_fp, c_review = 5.0, 1.0, 0.10
cost_rows = []
for threshold in np.arange(0.05, 0.96, 0.05):
    pred = (p_valid >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_valid, pred, labels=[0, 1]).ravel()
    reviewed = int(pred.sum())
    total_cost = c_fn * fn + c_fp * fp + c_review * reviewed
    cost_rows.append({"threshold": threshold, "reviewed": reviewed, "fn": fn, "fp": fp, "total_cost": total_cost})
cost_table = pd.DataFrame(cost_rows)
best_policy = cost_table.loc[cost_table["total_cost"].idxmin()]
print(cost_table.round(2).to_string(index=False))
print("\nIllustrative lowest-cost threshold:")
print(best_policy.to_string())


 threshold  reviewed  fn  fp  total_cost
      0.05       195   3 155       189.5
      0.10       179   4 140       177.9
      0.15       129   9  95       152.9
      0.20        76  13  46       118.6
      0.25        70  14  41       118.0
      0.30        53  16  26       111.3
      0.35        50  16  23       108.0
      0.40        44  18  19       113.4
      0.45        35  23  15       133.5
      0.50        31  25  13       141.1
      0.55        22  30   9       161.2
      0.60        10  36   3       184.0
      0.65         6  40   3       203.6
      0.70         6  40   3       203.6
      0.75         4  41   2       207.4
      0.80         4  41   2       207.4
      0.85         2  43   2       217.2
      0.90         2  43   2       217.2
      0.95         0  43   0       215.0

Illustrative lowest-cost threshold:
threshold       0.35
reviewed       50.00
fn             16.00
fp             23.00
total_cost    108.00


In [13]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
RocCurveDisplay.from_predictions(y_valid, p_valid, ax=axes[0], color="#0F8795", name=best_name)
PrecisionRecallDisplay.from_predictions(y_valid, p_valid, ax=axes[1], color="#D26455", name=best_name)
axes[0].set_title("Validation ROC curve")
axes[1].set_title("Validation precision–recall curve")
plt.tight_layout(); plt.show()


## 8. Calibration

Ranking asks whether higher-risk cases appear above lower-risk cases. Calibration
asks whether cases assigned probability 0.20 default at roughly 20% in a group. The
Brier score is

$$
Brier = \frac{1}{n} \sum_{i=1}^{n} (p_i-y_i)^2.
$$

Calibration does not prove causality, fairness, or profitability. It is one part of
the evidence required before a probability is used in a financial workflow.


In [14]:
observed, predicted = calibration_curve(y_valid, p_valid, n_bins=8, strategy="quantile")
fig, ax = plt.subplots()
ax.plot(predicted, observed, marker="o", color="#0F8795", label="model")
ax.plot([0, 1], [0, 1], linestyle="--", color="#D26455", label="perfect calibration")
ax.set_xlabel("Mean predicted probability")
ax.set_ylabel("Observed default rate")
ax.set_title(f"Reliability diagram: {best_name}")
ax.legend(); plt.tight_layout(); plt.show()
print("Validation Brier score:", round(brier_score_loss(y_valid, p_valid), 4))


Validation Brier score: 0.1441


## Worked checks (completed)

### 1. Probability by hand

For the worked borrower with η = 1.36, the sigmoid gives p = 1/(1 + exp(−1.36)) ≈ 0.796 and the odds are p/(1−p) ≈ 3.92. The positive utilisation and overdue coefficients increase the score; the intercept shifts the baseline.

### 2. Regularisation comparison

The executed cross-validation grid above reports validation ROC-AUC, PR-AUC, Brier score, log-loss, and non-zero coefficient count for each penalty. The selected model is the one chosen by the stated validation metric, not by the test sample.

### 3. Threshold policy

The executed threshold table compares c = 0.20, 0.35, 0.50, and 0.65. Lowering c creates more positive decisions and higher recall; raising c creates fewer positive decisions and higher precision. The final c must be justified by error costs and review capacity.

### 4. Model explanation

The target is default in the next outcome window. The score is η = β₀ + xᵀβ. The sigmoid maps η to p ∈ [0, 1]. Log-loss fits probabilities and penalises confident mistakes. A threshold converts p into an action, while leakage, calibration drift, and changing policy costs remain deployment limitations.


## Weekly assignment (unsolved)

Build and explain a logistic probability-of-default baseline on a documented financial dataset. Submit the target definition and information cutoff, an executed audit, the score/sigmoid/log-loss mathematics, a regularisation comparison, a threshold analysis, one coefficient interpretation, one limitation, and a reproducible notebook. Do not use the test labels to choose the model or threshold.

## Submission checklist

- executed notebook with the target definition, information cutoff, and data source visible;
- comparison table for Ridge, LASSO, and Elastic Net;
- one reliability or ROC/PR plot with metric interpretation;
- one threshold table with a stated cost or capacity assumption;
- one paragraph on leakage, calibration, and limitations.

Do not report a score without naming the sample, metric, threshold, and information cutoff.